# PS S6E6 - 038 Blend add037 TabICL check

Experiment: `exp_20260610_038_blend_add037_tabicl_check`

Purpose:
- Add `037 TabICL v2 as-is` to the current candidate pool.
- Check whether 037 adds useful low-correlation / error-complement signal to 033 / 036 GPU LogReg add035 / 036 blend add035 / 034default / 031 and core components.
- This is a blend/correlation diagnostic. Final candidates should come from safe methods (`avg`, `hc_nonnegative_raw`, `nm_softmax_raw`).

Current context:
- 033 blend add032: CV `0.9700400336552478`, Public LB `0.97043`.
- 036 GPU LogReg add035: CV `0.9700674063284508`, Public LB `0.97037`.
- 036 blend add035: CV `0.9700727843277738`, Public LB `0.97023`.
- 035 updated shared001 RealMLP: CV `0.9687410900866934`, Public LB `0.97012`.
- 037 TabICL v2 as-is: CV `0.9580770153777599`, Public LB `0.95920`.

Important:
- 037 is weak as a standalone model, but may still be useful only if it receives non-zero safe-blend weight.
- Do not treat in-sample LogReg/Ridge/ElasticNet rows as final candidates.


In [1]:
# ============================================================
# 0. Imports / Config
# ============================================================

import json
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import spearmanr

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression, RidgeClassifier, Ridge, ElasticNet

warnings.filterwarnings("ignore")

try:
    import yaml
except Exception:
    yaml = None

COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260610_038_blend_add037_tabicl_check"
SEED = 42

TRAIN_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/train.csv")
TEST_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/test.csv")
SAMPLE_SUB_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")

# Primary location. The loader below also scans /kaggle/input if a file is not found here.
NPY_DIR = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")
INPUT_ROOT = Path("/kaggle/input")

OUTDIR = Path("/kaggle/working") / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

ID_COL = "id"
TARGET_COL = "class"

MODEL_SPECS = [
    {"key": "015_realmlp_seed42", "exp_id": "exp_20260605_015_shared001_realmlp_as_is", "family": "RealMLP", "role": "stable_single_realmlp_backup", "oof": "oof_exp_20260605_015_shared001_realmlp_as_is_proba.npy", "pred": "pred_exp_20260605_015_shared001_realmlp_as_is_proba.npy", "cv": 0.9681693449222352, "public_lb": 0.96977},
    {"key": "017_realmlp_bias", "exp_id": "exp_20260606_017_bias_search_on_015_realmlp", "family": "RealMLP", "role": "previous_best_realmlp_bias_backup", "oof": "oof_exp_20260606_017_bias_search_on_015_realmlp_proba.npy", "pred": "pred_exp_20260606_017_bias_search_on_015_realmlp_proba.npy", "cv": 0.9683022653955233, "public_lb": 0.96985},
    {"key": "018_xgb_shared003", "exp_id": "exp_20260606_018_shared003_xgb_as_is", "family": "XGBoost", "role": "effective_blend_material", "oof": "oof_exp_20260606_018_shared003_xgb_as_is_proba.npy", "pred": "pred_exp_20260606_018_shared003_xgb_as_is_proba.npy", "cv": 0.965207986418628, "public_lb": 0.96578},
    {"key": "019_blend_best", "exp_id": "exp_20260607_019_blend_add018_xgb_check", "family": "Blend", "role": "previous_public_confirmed_best", "oof": "oof_exp_20260607_019_blend_add018_xgb_check_best_blend_proba.npy", "pred": "pred_exp_20260607_019_blend_add018_xgb_check_best_blend_proba.npy", "cv": 0.968872315017554, "public_lb": 0.97000},
    {"key": "020_blend_bias", "exp_id": "exp_20260607_020_bias_search_on_019_best_blend", "family": "Blend", "role": "cv_trusted_attack_reference", "oof": "oof_exp_20260607_020_bias_search_on_019_best_blend_proba.npy", "pred": "pred_exp_20260607_020_bias_search_on_019_best_blend_proba.npy", "cv": 0.9692240443617589, "public_lb": 0.96969},
    {"key": "024_seed_ensemble_blend", "exp_id": "exp_20260608_024_seed_ensemble_and_blend_check", "family": "Blend", "role": "seed_ensemble_blend_reference", "oof": "oof_exp_20260608_024_seed_ensemble_and_blend_check_best_blend_proba.npy", "pred": "pred_exp_20260608_024_seed_ensemble_and_blend_check_best_blend_proba.npy", "cv": 0.9694803109983217, "public_lb": 0.96958},
    {"key": "026_realmlp_v5", "exp_id": "exp_20260608_026_realmlp_v5_as_is", "family": "RealMLP", "role": "realmlp_v5_single_model_candidate", "oof": "oof_exp_20260608_026_realmlp_v5_as_is_proba.npy", "pred": "pred_exp_20260608_026_realmlp_v5_as_is_proba.npy", "cv": 0.9690389777981231, "public_lb": 0.96979},
    {"key": "027_blend_add026", "exp_id": "exp_20260608_027_blend_add026_realmlp_v5_check", "family": "Blend", "role": "previous_cv_trusted_slot", "oof": "oof_exp_20260608_027_blend_add026_realmlp_v5_check_best_blend_proba.npy", "pred": "pred_exp_20260608_027_blend_add026_realmlp_v5_check_best_blend_proba.npy", "cv": 0.9695425457477898, "public_lb": 0.96975},
    {"key": "028_cat_v3", "exp_id": "exp_20260608_028_cat_v3_as_is", "family": "CatBoost", "role": "catboost_v3_family_aux_material", "oof": "oof_exp_20260608_028_cat_v3_as_is_proba.npy", "pred": "pred_exp_20260608_028_cat_v3_as_is_proba.npy", "cv": 0.9688146470512534, "public_lb": 0.96969},
    {"key": "029_blend_add028", "exp_id": "exp_20260608_029_blend_add028_cat_v3_check", "family": "Blend", "role": "previous_best_backup", "oof": "oof_exp_20260608_029_blend_add028_cat_v3_check_best_blend_proba.npy", "pred": "pred_exp_20260608_029_blend_add028_cat_v3_check_best_blend_proba.npy", "cv": 0.9700023026377228, "public_lb": 0.970036},
    {"key": "030_ovr_xgb", "exp_id": "exp_20260608_030_ovr_xgb_as_is", "family": "XGBoost", "role": "low_weight_auxiliary_xgb_ovr_material", "oof": "oof_exp_20260608_030_ovr_xgb_as_is_proba.npy", "pred": "pred_exp_20260608_030_ovr_xgb_as_is_proba.npy", "cv": 0.9609971296999271, "public_lb": 0.96118},
    {"key": "031_blend_add030", "exp_id": "exp_20260608_031_blend_add030_ovr_xgb_check", "family": "Blend", "role": "public_confirmed_current_best_before_033", "oof": "oof_exp_20260608_031_blend_add030_ovr_xgb_check_best_blend_proba.npy", "pred": "pred_exp_20260608_031_blend_add030_ovr_xgb_check_best_blend_proba.npy", "cv": 0.9700333620193362, "public_lb": 0.97043},
    {"key": "032_ovr_tabm", "exp_id": "exp_20260609_032_ovr_tabm_as_is", "family": "TabM", "role": "tabm_ovr_family_aux_material", "oof": "oof_exp_20260609_032_ovr_tabm_as_is_proba.npy", "pred": "pred_exp_20260609_032_ovr_tabm_as_is_proba.npy", "cv": 0.9610105651284856, "public_lb": 0.96106, "tuned_cv": 0.9686168281485955, "public_lb_biased": 0.96895},
    {"key": "033_blend_add032", "exp_id": "exp_20260609_033_blend_add032_tabm_check", "family": "Blend", "role": "current_best_cv_public_balanced", "oof": "oof_exp_20260609_033_blend_add032_tabm_check_best_blend_proba.npy", "pred": "pred_exp_20260609_033_blend_add032_tabm_check_best_blend_proba.npy", "cv": 0.9700400336552478, "public_lb": 0.97043},
    {"key": "034_gpu_logreg_default", "exp_id": "exp_20260609_034_gpu_logreg_stacker_own", "family": "GPU_LogisticRegression_Stacker", "role": "stacker_backup_default", "oof": "oof_exp_20260609_034_gpu_logreg_stacker_own_proba.npy", "pred": "pred_exp_20260609_034_gpu_logreg_stacker_own_proba.npy", "cv": 0.9700373069292101, "public_lb": 0.97022, "raw_cv": 0.9699389938897909, "public_lb_raw": 0.97027},
    {"key": "035_shared001_updated", "exp_id": "exp_20260610_035_shared001_updated_realmlp_pytorch_as_is", "family": "RealMLP", "role": "updated_shared001_realmlp_aux_material", "oof": "oof_exp_20260610_035_shared001_updated_realmlp_pytorch_as_is_proba.npy", "pred": "pred_exp_20260610_035_shared001_updated_realmlp_pytorch_as_is_proba.npy", "cv": 0.9687410900866934, "public_lb": 0.97012},
    {"key": "036_gpu_logreg_add035", "exp_id": "exp_20260610_036_gpu_logreg_stacker_add035_own", "family": "GPU_LogisticRegression_Stacker", "role": "improved_stacker_backup_add035", "oof": "oof_exp_20260610_036_gpu_logreg_stacker_add035_own_proba.npy", "pred": "pred_exp_20260610_036_gpu_logreg_stacker_add035_own_proba.npy", "cv": 0.9700674063284508, "public_lb": 0.97037},
    {"key": "036_blend_add035", "exp_id": "exp_20260610_036_blend_add035_shared001_check", "family": "Blend", "role": "cv_best_but_public_weak_add035_blend", "oof": "oof_exp_20260610_036_blend_add035_shared001_check_best_blend_proba.npy", "pred": "pred_exp_20260610_036_blend_add035_shared001_check_best_blend_proba.npy", "cv": 0.9700727843277738, "public_lb": 0.97023},
    {"key": "037_tabicl_v2", "exp_id": "exp_20260610_037_tabicl_v2_as_is", "family": "TabICL", "role": "weak_tabicl_family_material_optional_corr", "oof": "oof_exp_20260610_037_tabicl_v2_as_is_proba.npy", "pred": "pred_exp_20260610_037_tabicl_v2_as_is_proba.npy", "cv": 0.9580770153777599, "public_lb": 0.95920},
]

# 038 should answer: does weak but different-family 037 add any complementary signal?
BLEND_SETS = {
    # Singles / current references
    "A_033_only": ["033_blend_add032"],
    "B_036_gpu_only": ["036_gpu_logreg_add035"],
    "C_036_blend_only": ["036_blend_add035"],
    "D_034_default_only": ["034_gpu_logreg_default"],
    "E_031_only": ["031_blend_add030"],
    "F_037_only": ["037_tabicl_v2"],
    "G_035_only": ["035_shared001_updated"],
    "H_030_only": ["030_ovr_xgb"],
    "I_032_only": ["032_ovr_tabm"],

    # Direct add037 checks against current candidates
    "J_033_037": ["033_blend_add032", "037_tabicl_v2"],
    "K_036_gpu_037": ["036_gpu_logreg_add035", "037_tabicl_v2"],
    "L_036_blend_037": ["036_blend_add035", "037_tabicl_v2"],
    "M_034_037": ["034_gpu_logreg_default", "037_tabicl_v2"],
    "N_031_037": ["031_blend_add030", "037_tabicl_v2"],
    "O_035_037": ["035_shared001_updated", "037_tabicl_v2"],
    "P_030_037": ["030_ovr_xgb", "037_tabicl_v2"],
    "Q_032_037": ["032_ovr_tabm", "037_tabicl_v2"],

    # Current best / stacker / core with 037
    "R_033_036gpu_037": ["033_blend_add032", "036_gpu_logreg_add035", "037_tabicl_v2"],
    "S_033_034_037": ["033_blend_add032", "034_gpu_logreg_default", "037_tabicl_v2"],
    "T_036gpu_035_037": ["036_gpu_logreg_add035", "035_shared001_updated", "037_tabicl_v2"],
    "U_033_036gpu_035_037": ["033_blend_add032", "036_gpu_logreg_add035", "035_shared001_updated", "037_tabicl_v2"],
    "V_036blend_036gpu_033_037": ["036_blend_add035", "036_gpu_logreg_add035", "033_blend_add032", "037_tabicl_v2"],

    # Core component diagnostics
    "W_027_026_028_030_032_035_037": ["027_blend_add026", "026_realmlp_v5", "028_cat_v3", "030_ovr_xgb", "032_ovr_tabm", "035_shared001_updated", "037_tabicl_v2"],
    "X_018_019_027_026_028_030_032_035_037": ["018_xgb_shared003", "019_blend_best", "027_blend_add026", "026_realmlp_v5", "028_cat_v3", "030_ovr_xgb", "032_ovr_tabm", "035_shared001_updated", "037_tabicl_v2"],

    # Broad diagnostics. Interpret weights carefully because composite blends and components coexist.
    "Y_033_036gpu_036blend_034_031_029_027_026_028_030_032_035_037": ["033_blend_add032", "036_gpu_logreg_add035", "036_blend_add035", "034_gpu_logreg_default", "031_blend_add030", "029_blend_add028", "027_blend_add026", "026_realmlp_v5", "028_cat_v3", "030_ovr_xgb", "032_ovr_tabm", "035_shared001_updated", "037_tabicl_v2"],
    "Z_full_015_017_018_019_020_024_026_027_028_029_030_031_032_033_034_035_036gpu_036blend_037": [
        "015_realmlp_seed42", "017_realmlp_bias", "018_xgb_shared003",
        "019_blend_best", "020_blend_bias", "024_seed_ensemble_blend",
        "026_realmlp_v5", "027_blend_add026", "028_cat_v3",
        "029_blend_add028", "030_ovr_xgb", "031_blend_add030",
        "032_ovr_tabm", "033_blend_add032", "034_gpu_logreg_default",
        "035_shared001_updated", "036_gpu_logreg_add035", "036_blend_add035", "037_tabicl_v2",
    ],
}

print("EXP_ID:", EXP_ID)
print("OUTDIR:", OUTDIR)
print("NPY_DIR:", NPY_DIR)
print("n_models:", len(MODEL_SPECS))
print("n_blend_sets:", len(BLEND_SETS))


EXP_ID: exp_20260610_038_blend_add037_tabicl_check
OUTDIR: /kaggle/working/exp_20260610_038_blend_add037_tabicl_check
NPY_DIR: /kaggle/input/datasets/mizushimatoshihiko/npy-files
n_models: 19
n_blend_sets: 26


In [2]:
# ============================================================
# 1. Utilities
# ============================================================

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)

def resolve_npy(filename):
    """Resolve an npy path. Prefer NPY_DIR, then scan /kaggle/input.

    This is intentionally more robust than 027 because 028 may be uploaded as a
    separate dataset from older npy files.
    """
    p = NPY_DIR / filename
    if p.exists():
        return p
    matches = list(INPUT_ROOT.rglob(filename)) if INPUT_ROOT.exists() else []
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        print(f"[WARN] multiple matches for {filename}. Using first:")
        for m in matches[:10]:
            print("  ", m)
        return matches[0]
    raise FileNotFoundError(f"Missing npy file: {filename}. Checked {NPY_DIR} and scanned {INPUT_ROOT}.")

def validate_proba(name, arr, n_rows, n_classes):
    assert arr.shape == (n_rows, n_classes), (name, arr.shape)
    assert np.isfinite(arr).all(), name
    assert np.allclose(arr.sum(axis=1), 1.0, atol=1e-4), name
    assert arr.min() >= -1e-7, (name, arr.min())
    assert arr.max() <= 1.0 + 1e-7, (name, arr.max())

def normalize_rows(p, eps=1e-12):
    p = np.asarray(p, dtype=np.float64)
    p = np.clip(p, eps, None)
    return p / p.sum(axis=1, keepdims=True)

def softmax_np(x, axis=1):
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / np.sum(ex, axis=axis, keepdims=True)

def proba_to_pred(p):
    return np.argmax(p, axis=1)

def flatten_proba(p):
    return np.asarray(p, dtype=float).reshape(-1)

def corr_pearson(a, b):
    a, b = flatten_proba(a), flatten_proba(b)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])

def corr_spearman(a, b):
    a, b = flatten_proba(a), flatten_proba(b)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(spearmanr(a, b).correlation)

def class_recalls(y_true, y_pred, class_names):
    out = {}
    for i, cls in enumerate(class_names):
        m = y_true == i
        out[f"recall_{cls}"] = float((y_pred[m] == i).mean()) if m.any() else float("nan")
    return out

def evaluate_proba(y_true, p, class_names):
    pred = proba_to_pred(p)
    res = {"balanced_accuracy": float(balanced_accuracy_score(y_true, pred))}
    res.update(class_recalls(y_true, pred, class_names))
    return res

def rank_normalize_matrix(p):
    out = np.zeros_like(p, dtype=np.float64)
    n = p.shape[0]
    for j in range(p.shape[1]):
        r = pd.Series(p[:, j]).rank(method="average").to_numpy()
        out[:, j] = (r - 1) / max(n - 1, 1)
    return normalize_rows(out)

def logit_transform(p, eps=1e-15):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

def build_meta_features(keys, proba_dict, mode):
    mats = []
    for k in keys:
        p = proba_dict[k]
        if mode == "raw":
            mats.append(p)
        elif mode == "rank":
            mats.append(rank_normalize_matrix(p))
        elif mode == "logit":
            mats.append(logit_transform(p))
        elif mode == "raw_logit":
            mats += [p, logit_transform(p)]
        elif mode == "raw_rank_logit":
            mats += [p, rank_normalize_matrix(p), logit_transform(p)]
        else:
            raise ValueError(mode)
    return np.concatenate(mats, axis=1)

def average_proba(keys, oof_dict, pred_dict=None):
    oof_avg = normalize_rows(np.mean([oof_dict[k] for k in keys], axis=0))
    pred_avg = None
    if pred_dict is not None:
        pred_avg = normalize_rows(np.mean([pred_dict[k] for k in keys], axis=0))
    return oof_avg, pred_avg

def onehot_y(y, n_classes):
    out = np.zeros((len(y), n_classes), dtype=np.float64)
    out[np.arange(len(y)), y] = 1.0
    return out

def hill_climb_weights(y_true, probas, steps=(0.1, 0.05, 0.02, 0.01, 0.005, 0.002), max_iter=300):
    n = len(probas)
    w = np.ones(n, dtype=float) / n
    cur_score = balanced_accuracy_score(y_true, normalize_rows(sum(w[i] * probas[i] for i in range(n))).argmax(axis=1))
    hist = [{"iter": 0, "score": float(cur_score), "weights": w.copy().tolist()}]
    for step in steps:
        improved = True
        it = 0
        while improved and it < max_iter:
            improved = False
            it += 1
            best_score, best_w = cur_score, w.copy()
            for i in range(n):
                for direction in [-1, 1]:
                    cand_w = w.copy()
                    cand_w[i] += direction * step
                    cand_w = np.clip(cand_w, 0.0, None)
                    if cand_w.sum() <= 0:
                        continue
                    cand_w /= cand_w.sum()
                    cand = normalize_rows(sum(cand_w[j] * probas[j] for j in range(n)))
                    score = balanced_accuracy_score(y_true, cand.argmax(axis=1))
                    if score > best_score + 1e-15:
                        best_score, best_w = score, cand_w.copy()
            if best_score > cur_score + 1e-15:
                cur_score, w = best_score, best_w
                improved = True
                hist.append({"iter": len(hist), "step": step, "score": float(cur_score), "weights": w.copy().tolist()})
    return w, float(cur_score), hist

def nm_softmax_weights(y_true, probas, n_restarts=5, maxiter=2500):
    probas = [np.asarray(p, dtype=np.float64) for p in probas]
    n = len(probas)

    def eval_from_theta(theta):
        w = np.exp(theta - np.max(theta))
        w = w / w.sum()
        p = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
        return balanced_accuracy_score(y_true, p.argmax(axis=1)), w

    def objective(theta):
        score, _ = eval_from_theta(theta)
        return -score

    inits = [np.zeros(n)]
    rng = np.random.default_rng(SEED)
    for _ in range(n_restarts - 1):
        inits.append(rng.normal(0, 0.25, size=n))

    best_score, best_w, best_res = -1, None, None
    for init in inits:
        res = minimize(objective, init, method="Nelder-Mead", options={"maxiter": maxiter, "xatol": 1e-8, "fatol": 1e-12})
        score, w = eval_from_theta(res.x)
        if score > best_score:
            best_score, best_w, best_res = score, w, res
    return best_w, float(best_score), best_res

def disagreement_and_error_overlap(y_true, p1, p2):
    pred1 = p1.argmax(axis=1)
    pred2 = p2.argmax(axis=1)
    wrong1 = pred1 != y_true
    wrong2 = pred2 != y_true
    both_wrong = wrong1 & wrong2
    either_wrong = wrong1 | wrong2
    return {
        "disagreement_rate": float((pred1 != pred2).mean()),
        "wrong_rate_left": float(wrong1.mean()),
        "wrong_rate_right": float(wrong2.mean()),
        "both_wrong_rate": float(both_wrong.mean()),
        "error_overlap_jaccard": float(both_wrong.sum() / max(either_wrong.sum(), 1)),
        "left_wrong_right_correct_rate": float((wrong1 & ~wrong2).mean()),
        "right_wrong_left_correct_rate": float((wrong2 & ~wrong1).mean()),
    }


In [3]:
# ============================================================
# 2. Load data and OOF/pred
# ============================================================

for p in [TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_SUB_PATH)

le = LabelEncoder()
y = le.fit_transform(train[TARGET_COL].astype(str))
class_names = le.classes_.tolist()
n_classes = len(class_names)
assert class_names == ["GALAXY", "QSO", "STAR"], class_names

oof, pred, resolved_specs = {}, {}, []
for spec in MODEL_SPECS:
    key = spec["key"]
    oof_path = resolve_npy(spec["oof"])
    pred_path = resolve_npy(spec["pred"])
    oof_arr = np.load(oof_path)
    pred_arr = np.load(pred_path)
    validate_proba(f"oof:{key}", oof_arr, len(train), n_classes)
    validate_proba(f"pred:{key}", pred_arr, len(test), n_classes)
    oof[key] = oof_arr.astype(np.float64)
    pred[key] = pred_arr.astype(np.float64)
    s = spec.copy()
    s["resolved_oof_path"] = str(oof_path)
    s["resolved_pred_path"] = str(pred_path)
    resolved_specs.append(s)
    print(f"loaded {key}: {oof_arr.shape} / {pred_arr.shape}")

model_keys = [s["key"] for s in resolved_specs]
print("class_names:", class_names)
print("loaded keys:", model_keys)


loaded 015_realmlp_seed42: (577347, 3) / (247435, 3)
loaded 017_realmlp_bias: (577347, 3) / (247435, 3)
loaded 018_xgb_shared003: (577347, 3) / (247435, 3)
loaded 019_blend_best: (577347, 3) / (247435, 3)
loaded 020_blend_bias: (577347, 3) / (247435, 3)
loaded 024_seed_ensemble_blend: (577347, 3) / (247435, 3)
loaded 026_realmlp_v5: (577347, 3) / (247435, 3)
loaded 027_blend_add026: (577347, 3) / (247435, 3)
loaded 028_cat_v3: (577347, 3) / (247435, 3)
loaded 029_blend_add028: (577347, 3) / (247435, 3)
loaded 030_ovr_xgb: (577347, 3) / (247435, 3)
loaded 031_blend_add030: (577347, 3) / (247435, 3)
loaded 032_ovr_tabm: (577347, 3) / (247435, 3)
loaded 033_blend_add032: (577347, 3) / (247435, 3)
loaded 034_gpu_logreg_default: (577347, 3) / (247435, 3)
loaded 035_shared001_updated: (577347, 3) / (247435, 3)
loaded 036_gpu_logreg_add035: (577347, 3) / (247435, 3)
loaded 036_blend_add035: (577347, 3) / (247435, 3)
loaded 037_tabicl_v2: (577347, 3) / (247435, 3)
class_names: ['GALAXY', 'QSO'

In [4]:
# ============================================================
# 3. Individual scores and pairwise diagnostics
# ============================================================

rows = []
for spec in resolved_specs:
    key = spec["key"]
    p = oof[key]
    y_pred = proba_to_pred(p)
    score = balanced_accuracy_score(y, y_pred)
    row = {
        "key": key,
        "exp_id": spec["exp_id"],
        "family": spec["family"],
        "role": spec["role"],
        "cv_from_memo": spec.get("cv", np.nan),
        "public_lb": spec.get("public_lb", np.nan),
        "public_lb_biased": spec.get("public_lb_biased", np.nan),
        "tuned_cv": spec.get("tuned_cv", np.nan),
        "recomputed_oof_cv": float(score),
        "delta_recomputed_minus_memo": float(score - spec.get("cv", score)),
    }
    row.update(class_recalls(y, y_pred, class_names))
    rows.append(row)

individual_df = pd.DataFrame(rows).sort_values("recomputed_oof_cv", ascending=False).reset_index(drop=True)
display(individual_df)
individual_df.to_csv(OUTDIR / "individual_scores.csv", index=False)

pair_rows = []
for a, b in combinations(model_keys, 2):
    diag = disagreement_and_error_overlap(y, oof[a], oof[b])
    pair_rows.append({
        "left": a,
        "right": b,
        "pearson_flat_proba": corr_pearson(oof[a], oof[b]),
        "spearman_flat_proba": corr_spearman(oof[a], oof[b]),
        **diag,
    })

pairwise_df = pd.DataFrame(pair_rows)
pairwise_df = pairwise_df.sort_values(["pearson_flat_proba", "error_overlap_jaccard"], ascending=[True, True]).reset_index(drop=True)
display(pairwise_df)
pairwise_df.to_csv(OUTDIR / "pairwise_oof_correlation.csv", index=False)

# Focus tables for 037 and current references.
def focus_pairwise(key, filename):
    df = pairwise_df[(pairwise_df["left"] == key) | (pairwise_df["right"] == key)].copy()
    df = df.sort_values("pearson_flat_proba").reset_index(drop=True)
    display(df)
    df.to_csv(OUTDIR / filename, index=False)
    return df

focus_037_df = focus_pairwise("037_tabicl_v2", "pairwise_oof_correlation_focus_037.csv")
focus_036_gpu_df = focus_pairwise("036_gpu_logreg_add035", "pairwise_oof_correlation_focus_036_gpu.csv")
focus_036_blend_df = focus_pairwise("036_blend_add035", "pairwise_oof_correlation_focus_036_blend.csv")
focus_035_df = focus_pairwise("035_shared001_updated", "pairwise_oof_correlation_focus_035.csv")
focus_033_df = focus_pairwise("033_blend_add032", "pairwise_oof_correlation_focus_033.csv")
focus_034_df = focus_pairwise("034_gpu_logreg_default", "pairwise_oof_correlation_focus_034.csv")
focus_031_df = focus_pairwise("031_blend_add030", "pairwise_oof_correlation_focus_031.csv")
focus_029_df = focus_pairwise("029_blend_add028", "pairwise_oof_correlation_focus_029.csv")


,key,exp_id,family,role,cv_from_memo,public_lb,public_lb_biased,tuned_cv,recomputed_oof_cv,delta_recomputed_minus_memo,recall_GALAXY,recall_QSO,recall_STAR
0,036_blend_add035,exp_20260610_036_blend_add035_shared001_check,Blend,cv_best_but_public_weak_add035_blend,0.970073,0.970230,NaN,NaN,0.970073,0.0,0.960777,0.976943,0.972499
1,036_gpu_logreg_add035,exp_20260610_036_gpu_logreg_stacker_add035_own,GPU_LogisticRegression_Stacker,improved_stacker_backup_add035,0.970067,0.970370,NaN,NaN,0.970067,0.0,0.958313,0.978701,0.973188
2,033_blend_add032,exp_20260609_033_blend_add032_tabm_check,Blend,current_best_cv_public_balanced,0.970040,0.970430,NaN,NaN,0.970040,0.0,0.961775,0.975773,0.972571
3,034_gpu_logreg_default,exp_20260609_034_gpu_logreg_stacker_own,GPU_LogisticRegression_Stacker,stacker_backup_default,0.970037,0.970220,NaN,NaN,0.970037,0.0,0.959794,0.977771,0.972547
4,031_blend_add030,exp_20260608_031_blend_add030_ovr_xgb_check,Blend,public_confirmed_current_best_before_033,0.970033,0.970430,NaN,NaN,0.970033,0.0,0.961738,0.975790,0.972571
5,029_blend_add028,exp_20260608_029_blend_add028_cat_v3_check,Blend,previous_best_backup,0.970002,0.970036,NaN,NaN,0.970002,0.0,0.961315,0.975867,0.972825
6,027_blend_add026,exp_20260608_027_blend_add026_realmlp_v5_check,Blend,previous_cv_trusted_slot,0.969543,0.969750,NaN,NaN,0.969543,0.0,0.961479,0.976439,0.970710
7,024_seed_ensemble_blend,exp_20260608_024_seed_ensemble_and_blend_check,Blend,seed_ensemble_blend_reference,0.969480,0.969580,NaN,NaN,0.969480,0.0,0.961956,0.976174,0.970311
8,020_blend_bias,exp_20260607_020_bias_search_on_019_best_blend,Blend,cv_trusted_attack_reference,0.969224,0.969690,NaN,NaN,0.969224,0.0,0.961137,0.976200,0.970335
9,026_realmlp_v5,exp_20260608_026_realmlp_v5_as_is,RealMLP,realmlp_v5_single_model_candidate,0.969039,0.969790,NaN,NaN,0.969039,0.0,0.959505,0.976431,0.971181


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,030_ovr_xgb,037_tabicl_v2,0.974477,0.953569,0.037044,0.028955,0.049665,0.020975,0.363871,0.007980,0.028690
1,032_ovr_tabm,037_tabicl_v2,0.974850,0.953750,0.037186,0.029062,0.049665,0.020963,0.362909,0.008099,0.028702
2,015_realmlp_seed42,037_tabicl_v2,0.979561,0.910951,0.031806,0.034388,0.049665,0.026350,0.456641,0.008038,0.023315
3,017_realmlp_bias,037_tabicl_v2,0.980194,0.901659,0.031480,0.035542,0.049665,0.027077,0.465809,0.008465,0.022588
4,035_shared001_updated,037_tabicl_v2,0.981919,0.878275,0.029999,0.034984,0.049665,0.027516,0.481598,0.007469,0.022150
...,...,...,...,...,...,...,...,...,...,...,...
166,015_realmlp_seed42,017_realmlp_bias,0.999895,0.997811,0.002175,0.034388,0.035542,0.033907,0.941244,0.000482,0.001635
167,034_gpu_logreg_default,036_gpu_logreg_add035,0.999938,0.998882,0.001436,0.034731,0.035419,0.034366,0.960358,0.000365,0.001053
168,029_blend_add028,033_blend_add032,0.999993,0.999794,0.000438,0.034083,0.033838,0.033749,0.987632,0.000334,0.000088
169,029_blend_add028,031_blend_add030,0.999993,0.999801,0.000421,0.034083,0.033858,0.033768,0.988140,0.000315,0.000090


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,030_ovr_xgb,037_tabicl_v2,0.974477,0.953569,0.037044,0.028955,0.049665,0.020975,0.363871,0.007980,0.028690
1,032_ovr_tabm,037_tabicl_v2,0.974850,0.953750,0.037186,0.029062,0.049665,0.020963,0.362909,0.008099,0.028702
2,015_realmlp_seed42,037_tabicl_v2,0.979561,0.910951,0.031806,0.034388,0.049665,0.026350,0.456641,0.008038,0.023315
3,017_realmlp_bias,037_tabicl_v2,0.980194,0.901659,0.031480,0.035542,0.049665,0.027077,0.465809,0.008465,0.022588
4,035_shared001_updated,037_tabicl_v2,0.981919,0.878275,0.029999,0.034984,0.049665,0.027516,0.481598,0.007469,0.022150
5,026_realmlp_v5,037_tabicl_v2,0.982068,0.807136,0.029921,0.035388,0.049665,0.027770,0.484791,0.007618,0.021895
6,018_xgb_shared003,037_tabicl_v2,0.982874,0.958619,0.028087,0.033536,0.049665,0.027713,0.499438,0.005823,0.021952
7,019_blend_best,037_tabicl_v2,0.983600,0.927088,0.029040,0.032528,0.049665,0.026752,0.482521,0.005776,0.022913
8,027_blend_add026,037_tabicl_v2,0.983756,0.872165,0.029002,0.034163,0.049665,0.027597,0.490775,0.006566,0.022068
9,024_seed_ensemble_blend,037_tabicl_v2,0.983788,0.933831,0.029021,0.033962,0.049665,0.027484,0.489542,0.006478,0.022181


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,036_gpu_logreg_add035,037_tabicl_v2,0.985368,0.955626,0.026473,0.035419,0.049665,0.029461,0.529644,0.005958,0.020204
1,030_ovr_xgb,036_gpu_logreg_add035,0.989202,0.937636,0.022671,0.028955,0.035419,0.020958,0.482726,0.007997,0.014461
2,032_ovr_tabm,036_gpu_logreg_add035,0.989481,0.946766,0.022555,0.029062,0.035419,0.021069,0.485318,0.007993,0.014350
3,018_xgb_shared003,036_gpu_logreg_add035,0.993900,0.961192,0.014658,0.033536,0.035419,0.027223,0.652320,0.006313,0.008196
4,015_realmlp_seed42,036_gpu_logreg_add035,0.996811,0.924964,0.010680,0.034388,0.035419,0.029677,0.739523,0.004711,0.005742
5,017_realmlp_bias,036_gpu_logreg_add035,0.996961,0.922292,0.010595,0.035542,0.035419,0.030294,0.744921,0.005248,0.005125
6,035_shared001_updated,036_gpu_logreg_add035,0.997797,0.900401,0.008695,0.034984,0.035419,0.030954,0.784642,0.004031,0.004465
7,026_realmlp_v5,036_gpu_logreg_add035,0.997834,0.867962,0.008359,0.035388,0.035419,0.031328,0.793533,0.004060,0.004091
8,019_blend_best,036_gpu_logreg_add035,0.998015,0.937168,0.008537,0.032528,0.035419,0.029776,0.780062,0.002752,0.005643
9,028_cat_v3,036_gpu_logreg_add035,0.998128,0.967264,0.008394,0.035277,0.035419,0.031239,0.791747,0.004037,0.004179


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,036_blend_add035,037_tabicl_v2,0.985093,0.904353,0.027160,0.034264,0.049665,0.028541,0.515292,0.005723,0.021124
1,030_ovr_xgb,036_blend_add035,0.990463,0.869145,0.020823,0.028955,0.034264,0.021310,0.508472,0.007645,0.012954
2,032_ovr_tabm,036_blend_add035,0.990711,0.879875,0.020667,0.029062,0.034264,0.021429,0.511472,0.007633,0.012835
3,018_xgb_shared003,036_blend_add035,0.994216,0.903492,0.014182,0.033536,0.034264,0.026889,0.657240,0.006648,0.007375
4,015_realmlp_seed42,036_blend_add035,0.997365,0.899824,0.009616,0.034388,0.034264,0.029623,0.759020,0.004765,0.004640
5,017_realmlp_bias,036_blend_add035,0.997437,0.908192,0.009705,0.035542,0.034264,0.030150,0.760297,0.005392,0.004114
6,035_shared001_updated,036_blend_add035,0.998337,0.880209,0.007484,0.034984,0.034264,0.030968,0.808968,0.004017,0.003296
7,026_realmlp_v5,036_blend_add035,0.998402,0.946615,0.007200,0.035388,0.034264,0.031309,0.816551,0.004079,0.002955
8,028_cat_v3,036_blend_add035,0.998477,0.915623,0.007612,0.035277,0.034264,0.031045,0.806479,0.004231,0.003218
9,019_blend_best,036_blend_add035,0.998490,0.911773,0.007292,0.032528,0.034264,0.029812,0.806183,0.002716,0.004451


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,035_shared001_updated,037_tabicl_v2,0.981919,0.878275,0.029999,0.034984,0.049665,0.027516,0.481598,0.007469,0.022150
1,030_ovr_xgb,035_shared001_updated,0.989101,0.885761,0.021524,0.028955,0.034984,0.021342,0.501037,0.007612,0.013642
2,032_ovr_tabm,035_shared001_updated,0.989571,0.890900,0.021147,0.029062,0.034984,0.021580,0.508157,0.007483,0.013404
3,018_xgb_shared003,035_shared001_updated,0.991856,0.896346,0.017037,0.033536,0.034984,0.025865,0.606367,0.007671,0.009119
4,028_cat_v3,035_shared001_updated,0.995089,0.900310,0.013491,0.035277,0.034984,0.028527,0.683544,0.006750,0.006457
5,035_shared001_updated,036_gpu_logreg_add035,0.997797,0.900401,0.008695,0.034984,0.035419,0.030954,0.784642,0.004031,0.004465
6,019_blend_best,035_shared001_updated,0.997824,0.897507,0.008600,0.032528,0.034984,0.029532,0.777545,0.002996,0.005453
7,034_gpu_logreg_default,035_shared001_updated,0.997882,0.900631,0.008432,0.034731,0.034984,0.030742,0.788809,0.003989,0.004242
8,015_realmlp_seed42,035_shared001_updated,0.997978,0.893795,0.008044,0.034388,0.034984,0.030753,0.796295,0.003636,0.004231
9,017_realmlp_bias,035_shared001_updated,0.998021,0.892394,0.008059,0.035542,0.034984,0.031309,0.798339,0.004233,0.003675


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,033_blend_add032,037_tabicl_v2,0.984870,0.865547,0.027770,0.033838,0.049665,0.028016,0.504916,0.005821,0.021649
1,030_ovr_xgb,033_blend_add032,0.990558,0.821940,0.020357,0.028955,0.033838,0.021344,0.514960,0.007611,0.012493
2,032_ovr_tabm,033_blend_add032,0.990801,0.830690,0.020230,0.029062,0.033838,0.021446,0.517361,0.007616,0.012391
3,018_xgb_shared003,033_blend_add032,0.993883,0.858653,0.014542,0.033536,0.033838,0.026513,0.648849,0.007024,0.007325
4,015_realmlp_seed42,033_blend_add032,0.997205,0.868130,0.009731,0.034388,0.033838,0.029355,0.755191,0.005033,0.004483
5,017_realmlp_bias,033_blend_add032,0.997258,0.881385,0.009920,0.035542,0.033838,0.029828,0.754149,0.005714,0.004010
6,033_blend_add032,035_shared001_updated,0.998127,0.842031,0.007886,0.033838,0.034984,0.030554,0.798407,0.003284,0.004431
7,019_blend_best,033_blend_add032,0.998300,0.878981,0.007554,0.032528,0.033838,0.029478,0.799127,0.003050,0.004360
8,020_blend_bias,033_blend_add032,0.998542,0.899930,0.007273,0.034489,0.033838,0.030592,0.810704,0.003897,0.003246
9,026_realmlp_v5,033_blend_add032,0.998543,0.976770,0.007065,0.035388,0.033838,0.031158,0.818500,0.004230,0.002679


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,034_gpu_logreg_default,037_tabicl_v2,0.985243,0.955774,0.026547,0.034731,0.049665,0.029078,0.525643,0.005653,0.020587
1,030_ovr_xgb,034_gpu_logreg_default,0.990145,0.939215,0.021476,0.028955,0.034731,0.021218,0.499613,0.007737,0.013514
2,032_ovr_tabm,034_gpu_logreg_default,0.990372,0.948525,0.021389,0.029062,0.034731,0.021310,0.501590,0.007753,0.013422
3,018_xgb_shared003,034_gpu_logreg_default,0.994361,0.962844,0.013974,0.033536,0.034731,0.027219,0.663108,0.006317,0.007512
4,015_realmlp_seed42,034_gpu_logreg_default,0.997000,0.924373,0.010287,0.034388,0.034731,0.029530,0.745898,0.004858,0.005201
5,017_realmlp_bias,034_gpu_logreg_default,0.997090,0.921749,0.010339,0.035542,0.034731,0.030076,0.748190,0.005466,0.004656
6,034_gpu_logreg_default,035_shared001_updated,0.997882,0.900631,0.008432,0.034731,0.034984,0.030742,0.788809,0.003989,0.004242
7,026_realmlp_v5,034_gpu_logreg_default,0.997888,0.866186,0.008113,0.035388,0.034731,0.031103,0.797168,0.004285,0.003629
8,028_cat_v3,034_gpu_logreg_default,0.998248,0.967845,0.007921,0.035277,0.034731,0.031125,0.800481,0.004152,0.003606
9,019_blend_best,034_gpu_logreg_default,0.998305,0.937727,0.007754,0.032528,0.034731,0.029819,0.796447,0.002709,0.004912


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,031_blend_add030,037_tabicl_v2,0.984873,0.865479,0.027770,0.033858,0.049665,0.028026,0.505009,0.005832,0.021639
1,030_ovr_xgb,031_blend_add030,0.990536,0.821836,0.020385,0.028955,0.033858,0.021341,0.514576,0.007614,0.012518
2,031_blend_add030,032_ovr_tabm,0.990777,0.830573,0.020262,0.033858,0.029062,0.021441,0.516912,0.012417,0.007621
3,018_xgb_shared003,031_blend_add030,0.993874,0.858577,0.014553,0.033536,0.033858,0.026518,0.648729,0.007018,0.007340
4,015_realmlp_seed42,031_blend_add030,0.997199,0.868061,0.009734,0.034388,0.033858,0.029364,0.755178,0.005025,0.004495
5,017_realmlp_bias,031_blend_add030,0.997253,0.881326,0.009913,0.035542,0.033858,0.029842,0.754368,0.005700,0.004017
6,031_blend_add030,035_shared001_updated,0.998123,0.841979,0.007883,0.033858,0.034984,0.030566,0.798543,0.003293,0.004418
7,019_blend_best,031_blend_add030,0.998293,0.878896,0.007571,0.032528,0.033858,0.029480,0.798761,0.003048,0.004379
8,020_blend_bias,031_blend_add030,0.998538,0.899859,0.007276,0.034489,0.033858,0.030600,0.810673,0.003888,0.003258
9,026_realmlp_v5,031_blend_add030,0.998540,0.976821,0.007079,0.035388,0.033858,0.031162,0.818219,0.004226,0.002697


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,029_blend_add028,037_tabicl_v2,0.984930,0.866857,0.027699,0.034083,0.049665,0.028181,0.507138,0.005903,0.021484
1,029_blend_add028,030_ovr_xgb,0.990184,0.822720,0.020717,0.034083,0.028955,0.021287,0.509853,0.012796,0.007668
2,029_blend_add028,032_ovr_tabm,0.990460,0.831713,0.020577,0.034083,0.029062,0.021396,0.512488,0.012687,0.007666
3,018_xgb_shared003,029_blend_add028,0.993841,0.859932,0.014596,0.033536,0.034083,0.026611,0.648927,0.006925,0.007472
4,015_realmlp_seed42,029_blend_add028,0.997290,0.870762,0.009627,0.034388,0.034083,0.029523,0.757994,0.004865,0.004561
5,017_realmlp_bias,029_blend_add028,0.997375,0.884010,0.009729,0.035542,0.034083,0.030039,0.758827,0.005503,0.004044
6,029_blend_add028,035_shared001_updated,0.998174,0.843194,0.007810,0.034083,0.034984,0.030709,0.800596,0.003374,0.004275
7,019_blend_best,029_blend_add028,0.998329,0.881267,0.007545,0.032528,0.034083,0.029604,0.799963,0.002924,0.004479
8,026_realmlp_v5,029_blend_add028,0.998564,0.976314,0.006989,0.035388,0.034083,0.031314,0.820654,0.004074,0.002770
9,020_blend_bias,029_blend_add028,0.998627,0.902219,0.007079,0.034489,0.034083,0.030808,0.815805,0.003681,0.003275


In [5]:
# ============================================================
# 4. Blend diagnostics
# ============================================================

blend_rows = {}
blend_probas = {}

def _weight_of(keys, weights, target_key):
    if weights is None or target_key not in keys:
        return None
    return float(dict(zip(keys, weights)).get(target_key, 0.0))

def record_blend(set_name, method, keys, oof_p, test_p=None, weights=None, extra=None):
    ev = evaluate_proba(y, oof_p, class_names)
    row = {
        "set_name": set_name,
        "method": method,
        "keys": ",".join(keys),
        "n_models": len(keys),
        "balanced_accuracy": ev["balanced_accuracy"],
        "includes_037": "037_tabicl_v2" in keys,
        "weight_037": _weight_of(keys, weights, "037_tabicl_v2"),
        "includes_036_gpu": "036_gpu_logreg_add035" in keys,
        "weight_036_gpu": _weight_of(keys, weights, "036_gpu_logreg_add035"),
        "includes_036_blend": "036_blend_add035" in keys,
        "weight_036_blend": _weight_of(keys, weights, "036_blend_add035"),
        "includes_035": "035_shared001_updated" in keys,
        "weight_035": _weight_of(keys, weights, "035_shared001_updated"),
        "includes_034": "034_gpu_logreg_default" in keys,
        "weight_034": _weight_of(keys, weights, "034_gpu_logreg_default"),
        "includes_033": "033_blend_add032" in keys,
        "weight_033": _weight_of(keys, weights, "033_blend_add032"),
        "includes_032": "032_ovr_tabm" in keys,
        "weight_032": _weight_of(keys, weights, "032_ovr_tabm"),
        "includes_031": "031_blend_add030" in keys,
        "weight_031": _weight_of(keys, weights, "031_blend_add030"),
        "includes_030": "030_ovr_xgb" in keys,
        "weight_030": _weight_of(keys, weights, "030_ovr_xgb"),
        "includes_029": "029_blend_add028" in keys,
        "weight_029": _weight_of(keys, weights, "029_blend_add028"),
        "includes_028": "028_cat_v3" in keys,
        "weight_028": _weight_of(keys, weights, "028_cat_v3"),
        "weights_json": json.dumps({k: float(w) for k, w in zip(keys, weights)}, ensure_ascii=False) if weights is not None else None,
    }
    row.update({k: v for k, v in ev.items() if k != "balanced_accuracy"})
    if extra:
        row.update(extra)
    blend_rows[(set_name, method)] = row
    if test_p is not None:
        blend_probas[(set_name, method)] = (normalize_rows(oof_p), normalize_rows(test_p))

for set_name, keys in BLEND_SETS.items():
    print("\n===", set_name, keys, "===")

    # 1) Simple average.
    avg_oof, avg_pred = average_proba(keys, oof, pred)
    record_blend(set_name, "avg", keys, avg_oof, avg_pred)

    # 2) Hill climb nonnegative raw probabilities.
    try:
        w, score, hist = hill_climb_weights(y, [oof[k] for k in keys])
        hc_oof = normalize_rows(sum(w[i] * oof[keys[i]] for i in range(len(keys))))
        hc_pred = normalize_rows(sum(w[i] * pred[keys[i]] for i in range(len(keys))))
        record_blend(set_name, "hc_nonnegative_raw", keys, hc_oof, hc_pred, weights=w, extra={"hc_score_internal": score, "hc_n_steps": len(hist)})
    except Exception as e:
        print(f"[WARN] HC failed: {set_name}: {e}")

    # 3) Nelder-Mead softmax weights.
    try:
        w, score, res = nm_softmax_weights(y, [oof[k] for k in keys])
        nm_oof = normalize_rows(sum(w[i] * oof[keys[i]] for i in range(len(keys))))
        nm_pred = normalize_rows(sum(w[i] * pred[keys[i]] for i in range(len(keys))))
        record_blend(set_name, "nm_softmax_raw", keys, nm_oof, nm_pred, weights=w, extra={"nm_score_internal": score, "nm_success": bool(res.success), "nm_message": str(res.message)})
    except Exception as e:
        print(f"[WARN] NM failed: {set_name}: {e}")

    # 4) In-sample meta models. Diagnostic only.
    for mode in ["raw", "rank", "logit", "raw_logit", "raw_rank_logit"]:
        X_meta = build_meta_features(keys, oof, mode)
        X_test_meta = build_meta_features(keys, pred, mode)
        try:
            lr = LogisticRegression(C=0.05, multi_class="multinomial", solver="lbfgs", max_iter=2000, random_state=SEED)
            lr.fit(X_meta, y)
            record_blend(set_name, f"logreg_{mode}", keys, lr.predict_proba(X_meta), lr.predict_proba(X_test_meta), extra={"C": 0.05, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] logreg failed: {set_name} {mode}: {e}")
        try:
            rc = RidgeClassifier(alpha=10.0, random_state=SEED)
            rc.fit(X_meta, y)
            record_blend(set_name, f"ridgeclf_{mode}", keys, softmax_np(rc.decision_function(X_meta)), softmax_np(rc.decision_function(X_test_meta)), extra={"alpha": 10.0, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] ridgeclf failed: {set_name} {mode}: {e}")
        try:
            y_oh = onehot_y(y, n_classes)
            rr = Ridge(alpha=10.0, random_state=SEED)
            rr.fit(X_meta, y_oh)
            record_blend(set_name, f"ridge_reg_{mode}", keys, normalize_rows(np.clip(rr.predict(X_meta), 1e-12, None)), normalize_rows(np.clip(rr.predict(X_test_meta), 1e-12, None)), extra={"alpha": 10.0, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] ridge_reg failed: {set_name} {mode}: {e}")
        try:
            y_oh = onehot_y(y, n_classes)
            oof_cols, test_cols = [], []
            for c in range(n_classes):
                en = ElasticNet(alpha=0.0005, l1_ratio=0.05, random_state=SEED, max_iter=2000)
                en.fit(X_meta, y_oh[:, c])
                oof_cols.append(en.predict(X_meta))
                test_cols.append(en.predict(X_test_meta))
            en_oof = normalize_rows(np.clip(np.vstack(oof_cols).T, 1e-12, None))
            en_pred = normalize_rows(np.clip(np.vstack(test_cols).T, 1e-12, None))
            record_blend(set_name, f"elasticnet_reg_{mode}", keys, en_oof, en_pred, extra={"alpha": 0.0005, "l1_ratio": 0.05, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] elasticnet_reg failed: {set_name} {mode}: {e}")

blend_df = pd.DataFrame(list(blend_rows.values())).sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(blend_df.head(200))
blend_df.to_csv(OUTDIR / "blend_diagnostics.csv", index=False)

safe_methods = ["avg", "hc_nonnegative_raw", "nm_softmax_raw"]
safe_blend_df = blend_df[blend_df["method"].isin(safe_methods)].copy().sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(safe_blend_df.head(80))
safe_blend_df.to_csv(OUTDIR / "safe_blend_diagnostics.csv", index=False)

focus_037_blend_df = safe_blend_df[safe_blend_df["includes_037"]].copy().sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(focus_037_blend_df.head(80))
focus_037_blend_df.to_csv(OUTDIR / "safe_blend_diagnostics_focus_037.csv", index=False)

focus_036_gpu_blend_df = safe_blend_df[safe_blend_df["includes_036_gpu"]].copy().sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(focus_036_gpu_blend_df.head(80))
focus_036_gpu_blend_df.to_csv(OUTDIR / "safe_blend_diagnostics_focus_036_gpu.csv", index=False)

focus_036_blend_blend_df = safe_blend_df[safe_blend_df["includes_036_blend"]].copy().sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(focus_036_blend_blend_df.head(80))
focus_036_blend_blend_df.to_csv(OUTDIR / "safe_blend_diagnostics_focus_036_blend.csv", index=False)

focus_035_blend_df = safe_blend_df[safe_blend_df["includes_035"]].copy().sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(focus_035_blend_df.head(80))
focus_035_blend_df.to_csv(OUTDIR / "safe_blend_diagnostics_focus_035.csv", index=False)

focus_033_blend_df = safe_blend_df[safe_blend_df["includes_033"]].copy().sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(focus_033_blend_df.head(80))
focus_033_blend_df.to_csv(OUTDIR / "safe_blend_diagnostics_focus_033.csv", index=False)

focus_034_blend_df = safe_blend_df[safe_blend_df["includes_034"]].copy().sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(focus_034_blend_df.head(80))
focus_034_blend_df.to_csv(OUTDIR / "safe_blend_diagnostics_focus_034.csv", index=False)

focus_031_blend_df = safe_blend_df[safe_blend_df["includes_031"]].copy().sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(focus_031_blend_df.head(80))
focus_031_blend_df.to_csv(OUTDIR / "safe_blend_diagnostics_focus_031.csv", index=False)



=== A_033_only ['033_blend_add032'] ===

=== B_036_gpu_only ['036_gpu_logreg_add035'] ===

=== C_036_blend_only ['036_blend_add035'] ===

=== D_034_default_only ['034_gpu_logreg_default'] ===

=== E_031_only ['031_blend_add030'] ===

=== F_037_only ['037_tabicl_v2'] ===

=== G_035_only ['035_shared001_updated'] ===

=== H_030_only ['030_ovr_xgb'] ===

=== I_032_only ['032_ovr_tabm'] ===

=== J_033_037 ['033_blend_add032', '037_tabicl_v2'] ===

=== K_036_gpu_037 ['036_gpu_logreg_add035', '037_tabicl_v2'] ===

=== L_036_blend_037 ['036_blend_add035', '037_tabicl_v2'] ===

=== M_034_037 ['034_gpu_logreg_default', '037_tabicl_v2'] ===

=== N_031_037 ['031_blend_add030', '037_tabicl_v2'] ===

=== O_035_037 ['035_shared001_updated', '037_tabicl_v2'] ===

=== P_030_037 ['030_ovr_xgb', '037_tabicl_v2'] ===

=== Q_032_037 ['032_ovr_tabm', '037_tabicl_v2'] ===

=== R_033_036gpu_037 ['033_blend_add032', '036_gpu_logreg_add035', '037_tabicl_v2'] ===

=== S_033_034_037 ['033_blend_add032', '034_gp

,set_name,method,keys,n_models,balanced_accuracy,includes_037,weight_037,includes_036_gpu,weight_036_gpu,includes_036_blend,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,V_036blend_036gpu_033_037,hc_nonnegative_raw,"036_blend_add035,036_gpu_logreg_add035,033_ble...",4,0.970102,True,0.0,True,0.476797,True,...,0.972825,0.970102,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,U_033_036gpu_035_037,hc_nonnegative_raw,"033_blend_add032,036_gpu_logreg_add035,035_sha...",4,0.970099,True,0.0,True,0.453662,False,...,0.972825,0.970099,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Y_033_036gpu_036blend_034_031_029_027_026_028_...,hc_nonnegative_raw,"033_blend_add032,036_gpu_logreg_add035,036_ble...",13,0.970087,True,0.0,True,0.217304,True,...,0.971508,0.970087,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Z_full_015_017_018_019_020_024_026_027_028_029...,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",19,0.970075,True,0.0,True,0.212287,True,...,0.972197,0.970075,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,R_033_036gpu_037,hc_nonnegative_raw,"033_blend_add032,036_gpu_logreg_add035,037_tab...",3,0.970073,True,0.0,True,0.490438,False,...,0.972741,0.970073,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,O_035_037,ridgeclf_raw_rank_logit,"035_shared001_updated,037_tabicl_v2",2,0.968068,True,NaN,False,NaN,False,...,0.962731,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,10.0000,NaN
196,O_035_037,ridge_reg_raw_rank_logit,"035_shared001_updated,037_tabicl_v2",2,0.968068,True,NaN,False,NaN,False,...,0.962731,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,10.0000,NaN
197,O_035_037,elasticnet_reg_raw_rank_logit,"035_shared001_updated,037_tabicl_v2",2,0.968027,True,NaN,False,NaN,False,...,0.962526,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,0.0005,0.05
198,S_033_034_037,ridge_reg_raw_rank_logit,"033_blend_add032,034_gpu_logreg_default,037_ta...",3,0.968017,True,NaN,False,NaN,False,...,0.958960,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,10.0000,NaN


,set_name,method,keys,n_models,balanced_accuracy,includes_037,weight_037,includes_036_gpu,weight_036_gpu,includes_036_blend,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,V_036blend_036gpu_033_037,hc_nonnegative_raw,"036_blend_add035,036_gpu_logreg_add035,033_ble...",4,0.970102,True,0.0,True,0.476797,True,...,0.972825,0.970102,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,U_033_036gpu_035_037,hc_nonnegative_raw,"033_blend_add032,036_gpu_logreg_add035,035_sha...",4,0.970099,True,0.0,True,0.453662,False,...,0.972825,0.970099,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Y_033_036gpu_036blend_034_031_029_027_026_028_...,hc_nonnegative_raw,"033_blend_add032,036_gpu_logreg_add035,036_ble...",13,0.970087,True,0.0,True,0.217304,True,...,0.971508,0.970087,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Z_full_015_017_018_019_020_024_026_027_028_029...,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",19,0.970075,True,0.0,True,0.212287,True,...,0.972197,0.970075,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,R_033_036gpu_037,hc_nonnegative_raw,"033_blend_add032,036_gpu_logreg_add035,037_tab...",3,0.970073,True,0.0,True,0.490438,False,...,0.972741,0.970073,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,H_030_only,avg,030_ovr_xgb,1,0.960997,False,NaN,False,NaN,False,...,0.937418,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
74,H_030_only,hc_nonnegative_raw,030_ovr_xgb,1,0.960997,False,NaN,False,NaN,False,...,0.937418,0.960997,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75,F_037_only,nm_softmax_raw,037_tabicl_v2,1,0.958077,True,1.0,False,NaN,False,...,0.961970,NaN,NaN,0.958077,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
76,F_037_only,hc_nonnegative_raw,037_tabicl_v2,1,0.958077,True,1.0,False,NaN,False,...,0.961970,0.958077,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,includes_037,weight_037,includes_036_gpu,weight_036_gpu,includes_036_blend,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,V_036blend_036gpu_033_037,hc_nonnegative_raw,"036_blend_add035,036_gpu_logreg_add035,033_ble...",4,0.970102,True,0.000000e+00,True,0.476797,True,...,0.972825,0.970102,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,U_033_036gpu_035_037,hc_nonnegative_raw,"033_blend_add032,036_gpu_logreg_add035,035_sha...",4,0.970099,True,0.000000e+00,True,0.453662,False,...,0.972825,0.970099,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Y_033_036gpu_036blend_034_031_029_027_026_028_...,hc_nonnegative_raw,"033_blend_add032,036_gpu_logreg_add035,036_ble...",13,0.970087,True,0.000000e+00,True,0.217304,True,...,0.971508,0.970087,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Z_full_015_017_018_019_020_024_026_027_028_029...,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",19,0.970075,True,0.000000e+00,True,0.212287,True,...,0.972197,0.970075,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,R_033_036gpu_037,hc_nonnegative_raw,"033_blend_add032,036_gpu_logreg_add035,037_tab...",3,0.970073,True,0.000000e+00,True,0.490438,False,...,0.972741,0.970073,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,L_036_blend_037,nm_softmax_raw,"036_blend_add035,037_tabicl_v2",2,0.970073,True,2.085848e-05,False,NaN,True,...,0.972499,NaN,NaN,0.970073,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
6,L_036_blend_037,hc_nonnegative_raw,"036_blend_add035,037_tabicl_v2",2,0.970073,True,0.000000e+00,False,NaN,True,...,0.972499,0.970073,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,K_036_gpu_037,nm_softmax_raw,"036_gpu_logreg_add035,037_tabicl_v2",2,0.970072,True,6.642756e-04,True,0.999336,False,...,0.973188,NaN,NaN,0.970072,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
8,R_033_036gpu_037,nm_softmax_raw,"033_blend_add032,036_gpu_logreg_add035,037_tab...",3,0.970068,True,7.298870e-09,True,0.999200,False,...,0.973188,NaN,NaN,0.970068,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
9,K_036_gpu_037,hc_nonnegative_raw,"036_gpu_logreg_add035,037_tabicl_v2",2,0.970067,True,0.000000e+00,True,1.000000,False,...,0.973188,0.970067,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,includes_037,weight_037,includes_036_gpu,weight_036_gpu,includes_036_blend,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,V_036blend_036gpu_033_037,hc_nonnegative_raw,"036_blend_add035,036_gpu_logreg_add035,033_ble...",4,0.970102,True,0.000000e+00,True,0.476797,True,...,0.972825,0.970102,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,U_033_036gpu_035_037,hc_nonnegative_raw,"033_blend_add032,036_gpu_logreg_add035,035_sha...",4,0.970099,True,0.000000e+00,True,0.453662,False,...,0.972825,0.970099,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Y_033_036gpu_036blend_034_031_029_027_026_028_...,hc_nonnegative_raw,"033_blend_add032,036_gpu_logreg_add035,036_ble...",13,0.970087,True,0.000000e+00,True,0.217304,True,...,0.971508,0.970087,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Z_full_015_017_018_019_020_024_026_027_028_029...,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",19,0.970075,True,0.000000e+00,True,0.212287,True,...,0.972197,0.970075,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,R_033_036gpu_037,hc_nonnegative_raw,"033_blend_add032,036_gpu_logreg_add035,037_tab...",3,0.970073,True,0.000000e+00,True,0.490438,False,...,0.972741,0.970073,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,K_036_gpu_037,nm_softmax_raw,"036_gpu_logreg_add035,037_tabicl_v2",2,0.970072,True,6.642756e-04,True,0.999336,False,...,0.973188,NaN,NaN,0.970072,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
6,R_033_036gpu_037,nm_softmax_raw,"033_blend_add032,036_gpu_logreg_add035,037_tab...",3,0.970068,True,7.298870e-09,True,0.999200,False,...,0.973188,NaN,NaN,0.970068,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
7,B_036_gpu_only,hc_nonnegative_raw,036_gpu_logreg_add035,1,0.970067,False,NaN,True,1.000000,False,...,0.973188,0.970067,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,B_036_gpu_only,nm_softmax_raw,036_gpu_logreg_add035,1,0.970067,False,NaN,True,1.000000,False,...,0.973188,NaN,NaN,0.970067,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
9,K_036_gpu_037,hc_nonnegative_raw,"036_gpu_logreg_add035,037_tabicl_v2",2,0.970067,True,0.000000e+00,True,1.000000,False,...,0.973188,0.970067,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,includes_037,weight_037,includes_036_gpu,weight_036_gpu,includes_036_blend,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,V_036blend_036gpu_033_037,hc_nonnegative_raw,"036_blend_add035,036_gpu_logreg_add035,033_ble...",4,0.970102,True,0.000000,True,0.476797,True,...,0.972825,0.970102,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Y_033_036gpu_036blend_034_031_029_027_026_028_...,hc_nonnegative_raw,"033_blend_add032,036_gpu_logreg_add035,036_ble...",13,0.970087,True,0.000000,True,0.217304,True,...,0.971508,0.970087,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Z_full_015_017_018_019_020_024_026_027_028_029...,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",19,0.970075,True,0.000000,True,0.212287,True,...,0.972197,0.970075,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,C_036_blend_only,nm_softmax_raw,036_blend_add035,1,0.970073,False,NaN,False,NaN,True,...,0.972499,NaN,NaN,0.970073,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
4,L_036_blend_037,nm_softmax_raw,"036_blend_add035,037_tabicl_v2",2,0.970073,True,0.000021,False,NaN,True,...,0.972499,NaN,NaN,0.970073,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
5,C_036_blend_only,hc_nonnegative_raw,036_blend_add035,1,0.970073,False,NaN,False,NaN,True,...,0.972499,0.970073,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,C_036_blend_only,avg,036_blend_add035,1,0.970073,False,NaN,False,NaN,True,...,0.972499,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,L_036_blend_037,hc_nonnegative_raw,"036_blend_add035,037_tabicl_v2",2,0.970073,True,0.000000,False,NaN,True,...,0.972499,0.970073,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,V_036blend_036gpu_033_037,nm_softmax_raw,"036_blend_add035,036_gpu_logreg_add035,033_ble...",4,0.969775,True,0.157671,True,0.358254,True,...,0.973260,NaN,NaN,0.969775,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
9,Z_full_015_017_018_019_020_024_026_027_028_029...,nm_softmax_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",19,0.969729,True,0.040510,True,0.066276,True,...,0.969525,NaN,NaN,0.969729,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,includes_037,weight_037,includes_036_gpu,weight_036_gpu,includes_036_blend,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,U_033_036gpu_035_037,hc_nonnegative_raw,"033_blend_add032,036_gpu_logreg_add035,035_sha...",4,0.970099,True,0.000000,True,0.453662,False,...,0.972825,0.970099,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Y_033_036gpu_036blend_034_031_029_027_026_028_...,hc_nonnegative_raw,"033_blend_add032,036_gpu_logreg_add035,036_ble...",13,0.970087,True,0.000000,True,0.217304,True,...,0.971508,0.970087,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Z_full_015_017_018_019_020_024_026_027_028_029...,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",19,0.970075,True,0.000000,True,0.212287,True,...,0.972197,0.970075,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,W_027_026_028_030_032_035_037,hc_nonnegative_raw,"027_blend_add026,026_realmlp_v5,028_cat_v3,030...",7,0.970006,True,0.000000,False,NaN,False,...,0.972862,0.970006,17.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,X_018_019_027_026_028_030_032_035_037,hc_nonnegative_raw,"018_xgb_shared003,019_blend_best,027_blend_add...",9,0.970001,True,0.000000,False,NaN,False,...,0.972426,0.970001,19.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,T_036gpu_035_037,nm_softmax_raw,"036_gpu_logreg_add035,035_shared001_updated,03...",3,0.969969,True,0.003278,True,0.767476,False,...,0.972729,NaN,NaN,0.969969,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
6,T_036gpu_035_037,hc_nonnegative_raw,"036_gpu_logreg_add035,035_shared001_updated,03...",3,0.969961,True,0.000000,True,0.709890,False,...,0.972584,0.969961,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Z_full_015_017_018_019_020_024_026_027_028_029...,nm_softmax_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",19,0.969729,True,0.040510,True,0.066276,True,...,0.969525,NaN,NaN,0.969729,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
8,Y_033_036gpu_036blend_034_031_029_027_026_028_...,nm_softmax_raw,"033_blend_add032,036_gpu_logreg_add035,036_ble...",13,0.969687,True,0.058995,True,0.079018,True,...,0.969332,NaN,NaN,0.969687,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
9,Z_full_015_017_018_019_020_024_026_027_028_029...,avg,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",19,0.969677,True,NaN,True,NaN,True,...,0.969392,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,includes_037,weight_037,includes_036_gpu,weight_036_gpu,includes_036_blend,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,V_036blend_036gpu_033_037,hc_nonnegative_raw,"036_blend_add035,036_gpu_logreg_add035,033_ble...",4,0.970102,True,0.000000e+00,True,0.476797,True,...,0.972825,0.970102,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,U_033_036gpu_035_037,hc_nonnegative_raw,"033_blend_add032,036_gpu_logreg_add035,035_sha...",4,0.970099,True,0.000000e+00,True,0.453662,False,...,0.972825,0.970099,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Y_033_036gpu_036blend_034_031_029_027_026_028_...,hc_nonnegative_raw,"033_blend_add032,036_gpu_logreg_add035,036_ble...",13,0.970087,True,0.000000e+00,True,0.217304,True,...,0.971508,0.970087,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Z_full_015_017_018_019_020_024_026_027_028_029...,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",19,0.970075,True,0.000000e+00,True,0.212287,True,...,0.972197,0.970075,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,R_033_036gpu_037,hc_nonnegative_raw,"033_blend_add032,036_gpu_logreg_add035,037_tab...",3,0.970073,True,0.000000e+00,True,0.490438,False,...,0.972741,0.970073,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,R_033_036gpu_037,nm_softmax_raw,"033_blend_add032,036_gpu_logreg_add035,037_tab...",3,0.970068,True,7.298870e-09,True,0.999200,False,...,0.973188,NaN,NaN,0.970068,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
6,S_033_034_037,hc_nonnegative_raw,"033_blend_add032,034_gpu_logreg_default,037_ta...",3,0.970041,True,0.000000e+00,False,NaN,False,...,0.972438,0.970041,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,A_033_only,nm_softmax_raw,033_blend_add032,1,0.970040,False,NaN,False,NaN,False,...,0.972571,NaN,NaN,0.970040,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
8,J_033_037,nm_softmax_raw,"033_blend_add032,037_tabicl_v2",2,0.970040,True,3.766323e-07,False,NaN,False,...,0.972571,NaN,NaN,0.970040,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
9,A_033_only,avg,033_blend_add032,1,0.970040,False,NaN,False,NaN,False,...,0.972571,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,includes_037,weight_037,includes_036_gpu,weight_036_gpu,includes_036_blend,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,Y_033_036gpu_036blend_034_031_029_027_026_028_...,hc_nonnegative_raw,"033_blend_add032,036_gpu_logreg_add035,036_ble...",13,0.970087,True,0.000000e+00,True,0.217304,True,...,0.971508,0.970087,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Z_full_015_017_018_019_020_024_026_027_028_029...,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",19,0.970075,True,0.000000e+00,True,0.212287,True,...,0.972197,0.970075,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,S_033_034_037,hc_nonnegative_raw,"033_blend_add032,034_gpu_logreg_default,037_ta...",3,0.970041,True,0.000000e+00,False,NaN,False,...,0.972438,0.970041,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,S_033_034_037,nm_softmax_raw,"033_blend_add032,034_gpu_logreg_default,037_ta...",3,0.970037,True,8.288179e-10,False,NaN,False,...,0.972547,NaN,NaN,0.970037,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
4,M_034_037,nm_softmax_raw,"034_gpu_logreg_default,037_tabicl_v2",2,0.970037,True,2.085848e-05,False,NaN,False,...,0.972547,NaN,NaN,0.970037,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
5,M_034_037,hc_nonnegative_raw,"034_gpu_logreg_default,037_tabicl_v2",2,0.970037,True,0.000000e+00,False,NaN,False,...,0.972547,0.970037,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,D_034_default_only,hc_nonnegative_raw,034_gpu_logreg_default,1,0.970037,False,NaN,False,NaN,False,...,0.972547,0.970037,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,D_034_default_only,avg,034_gpu_logreg_default,1,0.970037,False,NaN,False,NaN,False,...,0.972547,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,D_034_default_only,nm_softmax_raw,034_gpu_logreg_default,1,0.970037,False,NaN,False,NaN,False,...,0.972547,NaN,NaN,0.970037,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
9,Z_full_015_017_018_019_020_024_026_027_028_029...,nm_softmax_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",19,0.969729,True,4.050991e-02,True,0.066276,True,...,0.969525,NaN,NaN,0.969729,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,includes_037,weight_037,includes_036_gpu,weight_036_gpu,includes_036_blend,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,Y_033_036gpu_036blend_034_031_029_027_026_028_...,hc_nonnegative_raw,"033_blend_add032,036_gpu_logreg_add035,036_ble...",13,0.970087,True,0.000000,True,0.217304,True,...,0.971508,0.970087,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Z_full_015_017_018_019_020_024_026_027_028_029...,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",19,0.970075,True,0.000000,True,0.212287,True,...,0.972197,0.970075,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,N_031_037,nm_softmax_raw,"031_blend_add030,037_tabicl_v2",2,0.970034,True,0.000329,False,NaN,False,...,0.972571,NaN,NaN,0.970034,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
3,E_031_only,avg,031_blend_add030,1,0.970033,False,NaN,False,NaN,False,...,0.972571,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,E_031_only,hc_nonnegative_raw,031_blend_add030,1,0.970033,False,NaN,False,NaN,False,...,0.972571,0.970033,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,N_031_037,hc_nonnegative_raw,"031_blend_add030,037_tabicl_v2",2,0.970033,True,0.000000,False,NaN,False,...,0.972571,0.970033,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,E_031_only,nm_softmax_raw,031_blend_add030,1,0.970033,False,NaN,False,NaN,False,...,0.972571,NaN,NaN,0.970033,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
7,Z_full_015_017_018_019_020_024_026_027_028_029...,nm_softmax_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",19,0.969729,True,0.040510,True,0.066276,True,...,0.969525,NaN,NaN,0.969729,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
8,Y_033_036gpu_036blend_034_031_029_027_026_028_...,nm_softmax_raw,"033_blend_add032,036_gpu_logreg_add035,036_ble...",13,0.969687,True,0.058995,True,0.079018,True,...,0.969332,NaN,NaN,0.969687,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
9,Z_full_015_017_018_019_020_024_026_027_028_029...,avg,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",19,0.969677,True,NaN,True,NaN,True,...,0.969392,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# ============================================================
# 5. Save best safe blend
# ============================================================

safe_blend_df = safe_blend_df.sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
best_row = safe_blend_df.iloc[0].to_dict()
best_set_name = best_row["set_name"]
best_method = best_row["method"]
best_keys = best_row["keys"].split(",")

print("Best safe blend:")
print(json.dumps(best_row, indent=2, ensure_ascii=False))

if best_method == "avg":
    best_oof_proba, best_pred_proba = average_proba(best_keys, oof, pred)
else:
    weights_json = json.loads(best_row["weights_json"])
    w = np.array([weights_json[k] for k in best_keys], dtype=float)
    best_oof_proba = normalize_rows(sum(w[i] * oof[best_keys[i]] for i in range(len(best_keys))))
    best_pred_proba = normalize_rows(sum(w[i] * pred[best_keys[i]] for i in range(len(best_keys))))

validate_proba("best_oof_proba", best_oof_proba, len(train), n_classes)
validate_proba("best_pred_proba", best_pred_proba, len(test), n_classes)

best_oof_score = balanced_accuracy_score(y, best_oof_proba.argmax(axis=1))
print("best_oof_score:", best_oof_score)

best_oof_npy = OUTDIR / f"oof_{EXP_ID}_best_blend_proba.npy"
best_pred_npy = OUTDIR / f"pred_{EXP_ID}_best_blend_proba.npy"
np.save(best_oof_npy, best_oof_proba.astype(np.float32))
np.save(best_pred_npy, best_pred_proba.astype(np.float32))

best_labels = le.inverse_transform(best_pred_proba.argmax(axis=1))
submission = pd.DataFrame({ID_COL: test[ID_COL].values, TARGET_COL: best_labels})
assert submission.shape[0] == sample.shape[0]
assert submission.columns.tolist() == sample.columns.tolist()
assert submission[ID_COL].equals(sample[ID_COL])

best_submission_path = OUTDIR / f"submission_{EXP_ID}_best_blend.csv"
submission.to_csv(best_submission_path, index=False)

best_info = {
    "exp_id": EXP_ID,
    "best_set_name": best_set_name,
    "best_method": best_method,
    "best_keys": best_keys,
    "cv_score": float(best_oof_score),
    "includes_035": "035_shared001_updated" in best_keys,
    "weight_035": best_row.get("weight_035"),
    "includes_034": "034_gpu_logreg_default" in best_keys,
    "weight_034": best_row.get("weight_034"),
    "includes_033": "033_blend_add032" in best_keys,
    "weight_033": best_row.get("weight_033"),
    "includes_035": "035_shared001_updated" in best_keys,
    "weight_035": best_row.get("weight_035"),
    "includes_034": "034_gpu_logreg_default" in best_keys,
    "weight_034": best_row.get("weight_034"),
    "includes_033": "033_blend_add032" in best_keys,
    "weight_033": best_row.get("weight_033"),
    "includes_032": "032_ovr_tabm" in best_keys,
    "weight_032": best_row.get("weight_032"),
    "includes_031": "031_blend_add030" in best_keys,
    "weight_031": best_row.get("weight_031"),
    "includes_030": "030_ovr_xgb" in best_keys,
    "weight_030": best_row.get("weight_030"),
    "includes_029": "029_blend_add028" in best_keys,
    "weight_029": best_row.get("weight_029"),
    "includes_028": "028_cat_v3" in best_keys,
    "weight_028": best_row.get("weight_028"),
    "weights_json": best_row.get("weights_json"),
    "oof_path": str(best_oof_npy),
    "pred_path": str(best_pred_npy),
    "submission_path": str(best_submission_path),
    "row": best_row,
    "note": "Best safe blend excludes in-sample meta screening methods.",
}
save_json(best_info, OUTDIR / "best_blend_info.json")

saved_submission_df = pd.DataFrame([{
    "set_name": best_set_name,
    "method": best_method,
    "balanced_accuracy": float(best_oof_score),
    "includes_035": "035_shared001_updated" in best_keys,
    "weight_035": best_row.get("weight_035"),
    "includes_034": "034_gpu_logreg_default" in best_keys,
    "weight_034": best_row.get("weight_034"),
    "includes_033": "033_blend_add032" in best_keys,
    "weight_033": best_row.get("weight_033"),
    "includes_035": "035_shared001_updated" in best_keys,
    "weight_035": best_row.get("weight_035"),
    "includes_034": "034_gpu_logreg_default" in best_keys,
    "weight_034": best_row.get("weight_034"),
    "includes_033": "033_blend_add032" in best_keys,
    "weight_033": best_row.get("weight_033"),
    "includes_032": "032_ovr_tabm" in best_keys,
    "weight_032": best_row.get("weight_032"),
    "includes_031": "031_blend_add030" in best_keys,
    "weight_031": best_row.get("weight_031"),
    "submission": best_submission_path.name,
    "oof_proba": best_oof_npy.name,
    "pred_proba": best_pred_npy.name,
}])
display(saved_submission_df)
saved_submission_df.to_csv(OUTDIR / "saved_submission_candidates.csv", index=False)


Best safe blend:
{
  "set_name": "V_036blend_036gpu_033_037",
  "method": "hc_nonnegative_raw",
  "keys": "036_blend_add035,036_gpu_logreg_add035,033_blend_add032,037_tabicl_v2",
  "n_models": 4,
  "balanced_accuracy": 0.9701020196313722,
  "includes_037": true,
  "weight_037": 0.0,
  "includes_036_gpu": true,
  "weight_036_gpu": 0.4767966454741787,
  "includes_036_blend": true,
  "weight_036_blend": 0.3157274693587713,
  "includes_035": false,
  "weight_035": NaN,
  "includes_034": false,
  "weight_034": NaN,
  "includes_033": true,
  "weight_033": 0.20747588516705018,
  "includes_032": false,
  "weight_032": NaN,
  "includes_031": false,
  "weight_031": NaN,
  "includes_030": false,
  "weight_030": NaN,
  "includes_029": false,
  "weight_029": NaN,
  "includes_028": false,
  "weight_028": NaN,
  "weights_json": "{\"036_blend_add035\": 0.3157274693587713, \"036_gpu_logreg_add035\": 0.4767966454741787, \"033_blend_add032\": 0.20747588516705018, \"037_tabicl_v2\": 0.0}",
  "recall_GALAX

,set_name,method,balanced_accuracy,includes_035,weight_035,includes_034,weight_034,includes_033,weight_033,includes_032,weight_032,includes_031,weight_031,submission,oof_proba,pred_proba
0,V_036blend_036gpu_033_037,hc_nonnegative_raw,0.970102,False,NaN,False,NaN,True,0.207476,False,NaN,False,NaN,submission_exp_20260610_038_blend_add037_tabic...,oof_exp_20260610_038_blend_add037_tabicl_check...,pred_exp_20260610_038_blend_add037_tabicl_chec...


In [7]:
# ============================================================
# 6. Save summary / memo
# ============================================================

reference_scores = {
    "033_cv": 0.9700400336552478,
    "033_public_lb": 0.97043,
    "036_gpu_logreg_add035_cv": 0.9700674063284508,
    "036_gpu_logreg_add035_public_lb": 0.97037,
    "036_blend_add035_cv": 0.9700727843277738,
    "036_blend_add035_public_lb": 0.97023,
    "034_default_cv": 0.9700373069292101,
    "034_default_public_lb": 0.97022,
    "034_raw_cv": 0.9699389938897909,
    "034_raw_public_lb": 0.97027,
    "031_cv": 0.9700333620193362,
    "031_public_lb": 0.97043,
    "035_cv": 0.9687410900866934,
    "035_public_lb": 0.97012,
    "037_cv": 0.9580770153777599,
    "037_public_lb": 0.95920,
}

best_037_safe = focus_037_blend_df.iloc[0].to_dict() if len(focus_037_blend_df) else None
best_safe_weight_037 = best_info.get("weight_037")

judgement = {
    "reference_scores": reference_scores,
    "individual_best": {
        "key": individual_df.iloc[0]["key"],
        "cv": float(individual_df.iloc[0]["recomputed_oof_cv"]),
        "public_lb": float(individual_df.iloc[0]["public_lb"]),
    },
    "best_safe_blend": best_info,
    "best_safe_blend_with_037": best_037_safe,
    "main_questions": {
        "does_037_add_to_033": "Check J/R/S/U/V/Y safe methods and 037 weight.",
        "does_037_add_to_036_gpu": "Check K/R/T/U/V/Y safe methods and 037 weight.",
        "does_037_add_to_036_blend": "Check L/V/Y safe methods and 037 weight.",
        "is_037_too_weak_even_as_diversity_material": "Check focus_037 pairwise correlation and safe blend weight.",
        "should_promote_037": "Only if 037 receives meaningful non-zero safe blend weight and improves CV/Public LB.",
    },
    "automatic_view": {
        "best_safe_includes_037": bool(best_info.get("includes_037")),
        "best_safe_weight_037": best_safe_weight_037,
        "best_037_safe_cv": None if best_037_safe is None else float(best_037_safe["balanced_accuracy"]),
        "best_037_safe_method": None if best_037_safe is None else best_037_safe["method"],
        "best_037_safe_set": None if best_037_safe is None else best_037_safe["set_name"],
        "037_keep_rule": (
            "Keep only if 037 receives non-trivial safe blend weight or shows useful low-correlation behavior. "
            "If weight is near zero, hold 037 for optional correlation reference only."
        ),
    },
}
save_json(judgement, OUTDIR / "role_judgement.json")

summary = {
    "competition": COMPETITION,
    "exp_id": EXP_ID,
    "status": "completed",
    "purpose": "Blend/correlation check after adding 037 TabICL v2 as-is to 033/036_gpu/036_blend/034/031/035 candidate pool",
    "npy_dir": str(NPY_DIR),
    "model_keys": model_keys,
    "class_names": class_names,
    "resolved_specs": resolved_specs,
    "individual_scores": individual_df.to_dict(orient="records"),
    "pairwise_oof_correlation": pairwise_df.to_dict(orient="records"),
    "pairwise_oof_correlation_focus_037": focus_037_df.to_dict(orient="records"),
    "pairwise_oof_correlation_focus_036_gpu": focus_036_gpu_df.to_dict(orient="records"),
    "pairwise_oof_correlation_focus_036_blend": focus_036_blend_df.to_dict(orient="records"),
    "pairwise_oof_correlation_focus_035": focus_035_df.to_dict(orient="records"),
    "pairwise_oof_correlation_focus_033": focus_033_df.to_dict(orient="records"),
    "pairwise_oof_correlation_focus_034": focus_034_df.to_dict(orient="records"),
    "pairwise_oof_correlation_focus_031": focus_031_df.to_dict(orient="records"),
    "pairwise_oof_correlation_focus_029": focus_029_df.to_dict(orient="records"),
    "blend_diagnostics": blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics": safe_blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics_focus_037": focus_037_blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics_focus_036_gpu": focus_036_gpu_blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics_focus_036_blend": focus_036_blend_blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics_focus_035": focus_035_blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics_focus_033": focus_033_blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics_focus_034": focus_034_blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics_focus_031": focus_031_blend_df.to_dict(orient="records"),
    "best_blend_info": best_info,
    "saved_submission_candidates": saved_submission_df.to_dict(orient="records"),
    "role_judgement": judgement,
}
save_json(summary, OUTDIR / "blend_add037_tabicl_summary.json")

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "Blend check after adding 037 TabICL v2 as-is",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "created_at": "2026-06-10",
    },
    "objective": {
        "summary": (
            "Load OOF/pred probabilities from npy-files, add 037 TabICL v2 as-is, "
            "and decide whether this weak but different-family material adds complementary signal to current best blends/stackers."
        ),
        "success_criteria": [
            "load 015/017/018/019/020/024/026/027/028/029/030/031/032/033/034/035/036/037 OOF and pred npy files",
            "recompute individual balanced accuracy",
            "compute pairwise OOF correlations and focus table for 037",
            "evaluate add037 blend sets",
            "separate safe blends from in-sample meta screening",
            "save best safe blend OOF/pred npy",
            "save best safe blend submission",
        ],
    },
    "inputs": {"npy_dir": str(NPY_DIR), "models": resolved_specs, "blend_sets": BLEND_SETS},
    "outputs": {
        "individual_scores": "individual_scores.csv",
        "pairwise_oof_correlation": "pairwise_oof_correlation.csv",
        "pairwise_oof_correlation_focus_037": "pairwise_oof_correlation_focus_037.csv",
        "pairwise_oof_correlation_focus_036_gpu": "pairwise_oof_correlation_focus_036_gpu.csv",
        "pairwise_oof_correlation_focus_036_blend": "pairwise_oof_correlation_focus_036_blend.csv",
        "pairwise_oof_correlation_focus_035": "pairwise_oof_correlation_focus_035.csv",
        "blend_diagnostics": "blend_diagnostics.csv",
        "safe_blend_diagnostics": "safe_blend_diagnostics.csv",
        "safe_blend_diagnostics_focus_037": "safe_blend_diagnostics_focus_037.csv",
        "safe_blend_diagnostics_focus_036_gpu": "safe_blend_diagnostics_focus_036_gpu.csv",
        "safe_blend_diagnostics_focus_036_blend": "safe_blend_diagnostics_focus_036_blend.csv",
        "best_blend_info": "best_blend_info.json",
        "best_oof_proba": best_oof_npy.name,
        "best_pred_proba": best_pred_npy.name,
        "best_submission": best_submission_path.name,
        "summary": "blend_add037_tabicl_summary.json",
    },
    "judgement": {
        "status": "completed_pending_manual_review",
        "initial_view": (
            "037 has very low standalone CV/LB but belongs to a different family. "
            "Promote only if it improves safe blend CV or receives meaningful non-zero safe blend weight."
        ),
        "automatic_view": judgement["automatic_view"],
        "next_action": [
            "Review individual_scores.csv and confirm 037 recomputed OOF CV",
            "Review pairwise_oof_correlation_focus_037.csv",
            "Review safe_blend_diagnostics_focus_037.csv",
            "If best safe blend improves over 033/036_gpu and Public LB confirms, promote best_submission",
            "Do not treat in-sample LogReg/Ridge/ElasticNet rows as final candidates",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("Saved files:")
for p in sorted(OUTDIR.glob("*")):
    print(" -", p.name)


Saved files:
 - best_blend_info.json
 - blend_add037_tabicl_summary.json
 - blend_diagnostics.csv
 - individual_scores.csv
 - memo.yml
 - oof_exp_20260610_038_blend_add037_tabicl_check_best_blend_proba.npy
 - pairwise_oof_correlation.csv
 - pairwise_oof_correlation_focus_029.csv
 - pairwise_oof_correlation_focus_031.csv
 - pairwise_oof_correlation_focus_033.csv
 - pairwise_oof_correlation_focus_034.csv
 - pairwise_oof_correlation_focus_035.csv
 - pairwise_oof_correlation_focus_036_blend.csv
 - pairwise_oof_correlation_focus_036_gpu.csv
 - pairwise_oof_correlation_focus_037.csv
 - pred_exp_20260610_038_blend_add037_tabicl_check_best_blend_proba.npy
 - role_judgement.json
 - safe_blend_diagnostics.csv
 - safe_blend_diagnostics_focus_031.csv
 - safe_blend_diagnostics_focus_033.csv
 - safe_blend_diagnostics_focus_034.csv
 - safe_blend_diagnostics_focus_035.csv
 - safe_blend_diagnostics_focus_036_blend.csv
 - safe_blend_diagnostics_focus_036_gpu.csv
 - safe_blend_diagnostics_focus_037.csv
 